In [1]:
# https://medium.datadriveninvestor.com/automating-technical-analysis-pattern-recognition-bcbc85e4ec3c
## Run This Cell for Colab
!pip install nltk
!pip install pandas
!pip install beautifulsoup4

In [4]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

hdr = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.11 (KHTML, like Gecko) Chrome/23.0.1271.64 Safari/537.11',
       'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
       'Accept-Charset': 'ISO-8859-1,utf-8;q=0.7,*;q=0.3',
       'Accept-Encoding': 'none',
       'Accept-Language': 'en-US,en;q=0.8',
       'Connection': 'keep-alive'}

In [8]:
# Import libraries
import pandas as pd
import nltk
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from urllib.request import urlopen, Request
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

# Parameters 
n = 3 #the # of article headlines displayed per ticker
tickers = ['AAPL', 'TSLA', 'AMZN']

# Get Data
finwiz_url = 'https://finviz.com/quote.ashx?t='
news_tables = {}

for ticker in tickers:
    url = finwiz_url + ticker
    req = Request(url=url,headers=hdr) 
    resp = urlopen(req)    
    html = BeautifulSoup(resp, features="lxml")
    news_table = html.find(id='news-table')
    news_tables[ticker] = news_table

try:
    for ticker in tickers:
        df = news_tables[ticker]
        df_tr = df.findAll('tr')
    
        print ('\n')
        print ('Recent News Headlines for {}: '.format(ticker))
        
        for i, table_row in enumerate(df_tr):
            a_text = table_row.a.text
            td_text = table_row.td.text
            td_text = td_text.strip()
            print(a_text,'(',td_text,')')
            if i == n-1:
                break
except KeyError:
    pass

# Iterate through the news
parsed_news = []
for file_name, news_table in news_tables.items():
    for x in news_table.findAll('tr'):
        if x.a is None:
            continue
        text = x.a.get_text() 
        date_scrape = x.td.text.split()

        if len(date_scrape) == 1:
            time = date_scrape[0]
            
        else:
            date = date_scrape[0]
            time = date_scrape[1]

        ticker = file_name.split('_')[0]
        
        parsed_news.append([ticker, date, time, text])
        
# Sentiment Analysis
analyzer = SentimentIntensityAnalyzer()

columns = ['Ticker', 'Date', 'Time', 'Headline']
news = pd.DataFrame(parsed_news, columns=columns)
scores = news['Headline'].apply(analyzer.polarity_scores).tolist()

df_scores = pd.DataFrame(scores)
news = news.join(df_scores, rsuffix='_right')


# View Data 
news['Date'] = pd.to_datetime(news.Date).dt.date

unique_ticker = news['Ticker'].unique().tolist()
news_dict = {name: news.loc[news['Ticker'] == name] for name in unique_ticker}

values = []
for ticker in tickers: 
    dataframe = news_dict[ticker]
    dataframe = dataframe.set_index('Ticker')
    dataframe = dataframe.drop(columns = ['Headline'])
    print ('\n')
    print (dataframe.head())
    
    mean = round(dataframe['compound'].mean(), 2)
    values.append(mean)
    
df = pd.DataFrame(list(zip(tickers, values)), columns =['Ticker', 'Mean Sentiment']) 
df = df.set_index('Ticker')
df = df.sort_values('Mean Sentiment', ascending=False)
print ('\n')
print (df)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Ajay\AppData\Roaming\nltk_data...




Recent News Headlines for AAPL: 
50 Most Populated Countries In 2050 ( Jul-30-23 02:56PM )
Week Ahead: UK rates set to hit 15-year high ( 01:15PM )
Apple, Amazon earnings, July jobs report: What to know this week ( 10:43AM )


Recent News Headlines for TSLA: 
Musk draws heat from San Francisco over giant X logo ( Jul-30-23 04:53PM )
Tesla Dominance Pushes Major Carmaker to Exit Electric Vehicles ( 04:34PM )
Elon Musk Pledges His Heart to San Francisco ( 03:16PM )


Recent News Headlines for AMZN: 
Week Ahead: UK rates set to hit 15-year high ( Jul-30-23 01:15PM )
Apple, Amazon earnings, July jobs report: What to know this week ( 10:43AM )
Got $1,000? Buy These Hot Growth Stocks Before They Take Off ( 10:00AM )


              Date     Time  neg    neu    pos  compound
Ticker                                                  
AAPL    2023-07-30  02:56PM  0.0  1.000  0.000    0.0000
AAPL    2023-07-30  01:15PM  0.0  1.000  0.000    0.0000
AAPL    2023-07-30  10:43AM  0.0  0.855  0.145  